In [ ]:
# MATLAB section 1/7 for AnalysisExamples: Analysis Examples

# % Analysis Examples
# This is an example on the standard approach to fitting GLM models to
# spike train data. This data set was obtained at the Society For
# Neuroscience '08 Workshop on
# <http://www.sfn.org/index.aspx?pagename=ShortCourse3_2008 Workshop on Neural Signal Processing>
# Compare to analysis with
# <AnalysisExamples2.html Neural Spike Analysis Toolbox>

# Python translation bootstrap for this helpfile.

# Topic: AnalysisExamples
# Execution group: smoke
# Workflow family: analysis
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/AnalysisExamples.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "AnalysisExamples"
FAMILY = "analysis"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"AnalysisExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"AnalysisExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"AnalysisExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"AnalysisExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 5

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)


In [ ]:
# MATLAB section 2/7 for AnalysisExamples: Example 1: Tradition Preliminary Analysis

# % Example 1: Tradition Preliminary Analysis
#
# Script glm_part1.m
# MATLAB code to visualize data, fit a GLM model of the relation between
# spiking and the rat's position, and visualize this model for the
# Neuroinformatics GLM problem set.
# The code is initialized with an overly simple GLM model construction.
# Please improve it!
#
# load the rat trajectory and spiking data;
# MATLAB: close all;
# MATLAB: warning off;
# MATLAB: installPath = which('nSTAT_Install');
# MATLAB: if isempty(installPath)
# MATLAB:     error('AnalysisExamples:MissingInstallPath', ...
# MATLAB:         'Could not locate nSTAT_Install.m on the MATLAB path.');
# MATLAB: end
# MATLAB: glmDataPath = fullfile(fileparts(installPath), 'data', 'glm_data.mat');
# MATLAB: load(glmDataPath);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/7 for AnalysisExamples: Section

# %
# visualize the raw data
# MATLAB: figure;
# MATLAB: plot(xN,yN,x_at_spiketimes,y_at_spiketimes,'r.');
# MATLAB: axis tight square;
# MATLAB: xlabel('x position (m)'); ylabel('y position (m)');
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=30, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "AnalysisExamples.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 3, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/7 for AnalysisExamples: Section

# %
# fit a GLM model to the x and y positions.
# MATLAB: [b,dev,stats] = glmfit([xN yN (xN.^2-mean(xN.^2)) (yN.^2-mean(yN.^2)) (xN.*yN-mean(xN.*yN))],spikes_binned,'poisson');
# MATLAB: figure;
# MATLAB: errorbar(1:length(b), b, stats.se,'.');
# MATLAB: xticks=1:length(b);
# MATLAB: xtickLabels= {'baseline','x','y','x^2','y^2','x*y'};
# MATLAB: set(gca,'xtick',xticks,'xtickLabel',xtickLabels);
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=38, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "AnalysisExamples_01.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 4, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/7 for AnalysisExamples: Section

# %
# visualize your model
# construct a grid of positions to plot the model against...
# MATLAB: figure;
# MATLAB: [x_new,y_new]=meshgrid(-1:.1:1);
# MATLAB: y_new = flipud(y_new);
# MATLAB: x_new = fliplr(x_new);
#
# compute lambda for each point on this grid using the GLM model
# MATLAB: lambda = exp(b(1) + b(2)*x_new + b(3)*y_new + b(4)*x_new.^2 + b(5)*y_new.^2 + b(6)*x_new.*y_new);
# MATLAB: lambda((x_new.^2+y_new.^2>1))=nan;
#
# plot lambda as a function position over this grid
# MATLAB: h_mesh = mesh(x_new,y_new,lambda,'AlphaData',0);
# MATLAB: get(h_mesh,'AlphaData');
# MATLAB: set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.8,'EdgeColor','b');
# MATLAB: hold on;
# MATLAB: plot3(cos(-pi:1e-2:pi),sin(-pi:1e-2:pi),zeros(size(-pi:1e-2:pi))); hold on;
# MATLAB: plot(xN,yN,x_at_spiketimes,y_at_spiketimes,'r.');
# MATLAB: axis tight square;
# MATLAB: xlabel('x position (m)'); ylabel('y position (m)');
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=5, matlab_line_number=47, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "AnalysisExamples_02.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 5, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/7 for AnalysisExamples: Section

# %
# Compare a linear model versus a Gaussian GLM model.
# MATLAB: [b_lin,dev_lin,stats_lin] = glmfit([xN yN],spikes_binned,'poisson');
# MATLAB: [b_quad,dev_quad,stats_quad] = glmfit([xN yN xN.^2 yN.^2 xN.*yN],spikes_binned,'poisson');
#
# MATLAB: lambdaEst_lin = exp( b_lin(1) + b_lin(2)*xN+b_lin(3)*yN);  % based on our GLM model with the log "link function"
# MATLAB: lambdaEst_quad = exp( b_quad(1) + b_quad(2)*xN+b_quad(3)*yN+b_quad(4)*xN.^2 +b_quad(5)*yN.^2 +b_quad(6)*xN.*yN);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 6

_ = section_index


In [ ]:
# MATLAB section 7/7 for AnalysisExamples: Section

# %
# Make the KS Plot
#
# *******  K-S Plot  *******************
# graph the K-S plot and confidence intervals for the K-S statistic
#
# first generate the conditional intensity at each timestep
# ** Adjust the below line according to your choice of model.
# remember to include a column of ones to multiply the default constant GLM parameter beta_0**
#
# Use your parameter estimates (b) from glmfit along
# with the covariates you used (xN, yN, ...)
#
# MATLAB: lambdaEst=[lambdaEst_lin, lambdaEst_quad];
# MATLAB: timestep = 1;
# MATLAB: lambdaInt = 0;
# MATLAB: j=0;
# MATLAB: KS=[];
# MATLAB: for t=1:length(spikes_binned)
# MATLAB:     lambdaInt = lambdaInt + lambdaEst(t,:)*timestep;
# MATLAB:     if (spikes_binned(t))
# MATLAB:         j = j + 1;
# MATLAB:         KS(j,:) = 1-exp(-lambdaInt);
# MATLAB:         lambdaInt = [0 0];
# MATLAB:     end
# MATLAB: end
# MATLAB: KSSorted = sort( KS );
# MATLAB: N = length( KSSorted);
# MATLAB: figure;
# MATLAB: plot( ([1:N]-.5)/N, KSSorted, 0:.01:1,0:.01:1, 'g',0:.01:1, [0:.01:1]+1.36/sqrt(N), 'r', 0:.01:1,[0:.01:1]-1.36/sqrt(N), 'r' );
# MATLAB: axis( [0 1 0 1] );
# MATLAB: xlabel('Uniform CDF');
# MATLAB: ylabel('Empirical CDF of Rescaled ISIs');
# MATLAB: title('KS Plot with 95% Confidence Intervals');
# MATLAB: legend('Linear','Quadratic');

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=103, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "AnalysisExamples_03.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 7, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #5 for AnalysisExamples>
fig = FIGURE_TRACKER.new_figure(section_index=7, matlab_line_number=75, matlab_snippet="<synthetic MATLAB figure event #5 for AnalysisExamples>")
ref_image = (MATLAB_HELP_ROOT / "AnalysisExamples_04.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 7, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "warning off;",
    "installPath = which('nSTAT_Install');",
    "if isempty(installPath)",
    "error('AnalysisExamples:MissingInstallPath', ...",
    "'Could not locate nSTAT_Install.m on the MATLAB path.');",
    "end",
    "glmDataPath = fullfile(fileparts(installPath), 'data', 'glm_data.mat');",
    "load(glmDataPath);",
    "figure;",
    "plot(xN,yN,x_at_spiketimes,y_at_spiketimes,'r.');",
    "axis tight square;",
    "xlabel('x position (m)'); ylabel('y position (m)');",
    "[b,dev,stats] = glmfit([xN yN (xN.^2-mean(xN.^2)) (yN.^2-mean(yN.^2)) (xN.*yN-mean(xN.*yN))],spikes_binned,'poisson');",
    "figure;",
    "errorbar(1:length(b), b, stats.se,'.');",
    "xticks=1:length(b);",
    "xtickLabels= {'baseline','x','y','x^2','y^2','x*y'};",
    "set(gca,'xtick',xticks,'xtickLabel',xtickLabels);",
    "figure;",
    "[x_new,y_new]=meshgrid(-1:.1:1);",
    "y_new = flipud(y_new);",
    "x_new = fliplr(x_new);",
    "lambda = exp(b(1) + b(2)*x_new + b(3)*y_new + b(4)*x_new.^2 + b(5)*y_new.^2 + b(6)*x_new.*y_new);",
    "lambda((x_new.^2+y_new.^2>1))=nan;",
    "h_mesh = mesh(x_new,y_new,lambda,'AlphaData',0);",
    "get(h_mesh,'AlphaData');",
    "set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.8,'EdgeColor','b');",
    "hold on;",
    "plot3(cos(-pi:1e-2:pi),sin(-pi:1e-2:pi),zeros(size(-pi:1e-2:pi))); hold on;",
    "plot(xN,yN,x_at_spiketimes,y_at_spiketimes,'r.');",
    "axis tight square;",
    "xlabel('x position (m)'); ylabel('y position (m)');",
    "[b_lin,dev_lin,stats_lin] = glmfit([xN yN],spikes_binned,'poisson');",
    "[b_quad,dev_quad,stats_quad] = glmfit([xN yN xN.^2 yN.^2 xN.*yN],spikes_binned,'poisson');",
    "lambdaEst_lin = exp( b_lin(1) + b_lin(2)*xN+b_lin(3)*yN);  % based on our GLM model with the log \"link function\"",
    "lambdaEst_quad = exp( b_quad(1) + b_quad(2)*xN+b_quad(3)*yN+b_quad(4)*xN.^2 +b_quad(5)*yN.^2 +b_quad(6)*xN.*yN);",
    "lambdaEst=[lambdaEst_lin, lambdaEst_quad];",
    "timestep = 1;",
    "lambdaInt = 0;",
    "j=0;",
    "KS=[];",
    "for t=1:length(spikes_binned)",
    "lambdaInt = lambdaInt + lambdaEst(t,:)*timestep;",
    "if (spikes_binned(t))",
    "j = j + 1;",
    "KS(j,:) = 1-exp(-lambdaInt);",
    "lambdaInt = [0 0];",
    "end",
    "end",
    "KSSorted = sort( KS );",
    "N = length( KSSorted);",
    "figure;",
    "plot( ([1:N]-.5)/N, KSSorted, 0:.01:1,0:.01:1, 'g',0:.01:1, [0:.01:1]+1.36/sqrt(N), 'r', 0:.01:1,[0:.01:1]-1.36/sqrt(N), 'r' );",
    "axis( [0 1 0 1] );",
    "xlabel('Uniform CDF');",
    "ylabel('Empirical CDF of Rescaled ISIs');",
    "title('KS Plot with 95% Confidence Intervals');",
    "legend('Linear','Quadratic');"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for AnalysisExamples.")

# AnalysisExamples: spatial firing-rate modeling with x-y covariates.
n_t = 4500
dt = 0.01
time = np.arange(n_t) * dt

xy = np.zeros((n_t, 2), dtype=float)
xy[0] = np.array([45.0, 55.0])
vel = np.zeros(2, dtype=float)
for t in range(1, n_t):
    vel = 0.92 * vel + 2.0 * rng.normal(size=2)
    xy[t] = np.clip(xy[t - 1] + vel, 0.0, 100.0)

xc, yc, sigma = 62.0, 38.0, 16.0
r2 = (xy[:, 0] - xc) ** 2 + (xy[:, 1] - yc) ** 2
true_rate = 1.2 + 18.0 * np.exp(-0.5 * r2 / (sigma**2))
counts = rng.poisson(true_rate * dt)

X_lin = np.column_stack([xy[:, 0], xy[:, 1]])
fit_lin = Analysis.fit_glm(X=X_lin, y=counts, fit_type="poisson", dt=dt)
est_rate = fit_lin.predict(X_lin)

grid = np.linspace(0.0, 100.0, 35)
gx, gy = np.meshgrid(grid, grid, indexing="xy")
Xg = np.column_stack([gx.ravel(), gy.ravel()])
true_map = 1.2 + 18.0 * np.exp(-0.5 * (((Xg[:, 0] - xc) ** 2 + (Xg[:, 1] - yc) ** 2) / (sigma**2)))
est_map = fit_lin.predict(Xg)

spike_mask = counts > 0

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].plot(xy[:, 0], xy[:, 1], color="0.75", linewidth=0.8)
axes[0, 0].scatter(xy[spike_mask, 0], xy[spike_mask, 1], s=5, c="tab:red", alpha=0.6)
axes[0, 0].set_title(f"{TOPIC}: trajectory and spikes")
axes[0, 0].set_xlabel("x")
axes[0, 0].set_ylabel("y")
axes[0, 0].set_aspect("equal", adjustable="box")

im1 = axes[0, 1].imshow(true_map.reshape(grid.size, grid.size), origin="lower", extent=[0, 100, 0, 100], cmap="jet")
axes[0, 1].set_title("True place field")
axes[0, 1].set_xlabel("x")
axes[0, 1].set_ylabel("y")
fig.colorbar(im1, ax=axes[0, 1], fraction=0.04, pad=0.03)

im2 = axes[1, 0].imshow(est_map.reshape(grid.size, grid.size), origin="lower", extent=[0, 100, 0, 100], cmap="jet")
axes[1, 0].set_title("Estimated linear model field")
axes[1, 0].set_xlabel("x")
axes[1, 0].set_ylabel("y")
fig.colorbar(im2, ax=axes[1, 0], fraction=0.04, pad=0.03)

axes[1, 1].plot(time[:800], true_rate[:800], label="true", linewidth=1.1)
axes[1, 1].plot(time[:800], est_rate[:800], label="estimated", linewidth=1.0)
axes[1, 1].set_title("Rate trace (first 8 s)")
axes[1, 1].set_xlabel("time [s]")
axes[1, 1].set_ylabel("Hz")
axes[1, 1].legend(loc="upper right")

plt.tight_layout()
plt.show()

rmse_rate = float(np.sqrt(np.mean((est_rate - true_rate) ** 2)))
print("rmse_rate", rmse_rate, "aic", float(fit_lin.aic()))
assert np.isfinite(rmse_rate)
assert rmse_rate < 8.0

CHECKPOINT_METRICS = {
    "rmse_rate": float(rmse_rate),
    "mean_true_rate": float(np.mean(true_rate)),
}
CHECKPOINT_LIMITS = {
    "rmse_rate": (0.0, 8.0),
    "mean_true_rate": (0.2, 30.0),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
